# Completed T4 session
This is a transcript of the executed Colab cells, copied from their rendered sources and visible outputs after the runtime disconnected. Original execution counts were unavailable. Successful download widgets are represented by their visible text; the complete exported files and SHA-256 manifest are checked in separately. Failed outputs are retained.
Training used `55bf7f4`; the verification-only retry used `b21a349` and the same saved adapter. The final verdict is **DON'T SHIP**.


In [ ]:
import sys, subprocess, datetime, pathlib
print(datetime.datetime.now(datetime.timezone.utc).isoformat())
print(sys.version)
p = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(p.stdout or p.stderr)
print("exit:", p.returncode)


2026-10-04T22:27:51.324735+00:00
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sun Oct  4 22:27:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |             

In [ ]:
import pathlib, subprocess, sys, datetime
bootstrap = pathlib.Path("/content/bootstrap-evidence")
bootstrap.mkdir(exist_ok=True)
(bootstrap / "environment.txt").write_text(sys.version + "\n")
(bootstrap / "nvidia-smi.raw.log").write_bytes(subprocess.check_output(["nvidia-smi"]))
argv = [sys.executable, "-m", "pip", "install", "soup-cli[train]==0.75.2", "transformers==5.16.1", "trl==0.29.0", "peft==0.20.0", "pytest", "pyarrow", "reportlab", "pypdf"]
print("Installing:", argv)
with (bootstrap / "install.raw.log").open("wb") as raw, (bootstrap / "install.timestamped.log").open("w") as log:
    process = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in iter(process.stdout.readline, b""):
        raw.write(line); raw.flush()
        text = "[" + datetime.datetime.now(datetime.timezone.utc).isoformat() + "] " + line.decode(errors="replace")
        log.write(text); log.flush()
        print(text, end="")
    code = process.wait()
    process.stdout.close()
print("Installation exit:", code)
assert code == 0


Installing: ['/usr/bin/python3', '-m', 'pip', 'install', 'soup-cli[train]==0.75.2', 'transformers==5.16.1', 'trl==0.29.0', 'peft==0.20.0', 'pytest', 'pyarrow', 'reportlab', 'pypdf']
[2026-10-04T22:30:17.989207+00:00] Collecting soup-cli==0.75.2 (from soup-cli[train]==0.75.2)
[2026-10-04T22:30:18.030006+00:00]   Downloading soup_cli-0.75.2-py3-none-any.whl.metadata (29 kB)
[2026-10-04T22:30:18.262396+00:00] Collecting transformers==5.16.1
[2026-10-04T22:30:18.275026+00:00]   Downloading transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
[2026-10-04T22:30:18.456214+00:00] Collecting trl==0.29.0
[2026-10-04T22:30:18.471495+00:00]   Downloading trl-0.29.0-py3-none-any.whl.metadata (11 kB)
[2026-10-04T22:30:18.540160+00:00] Collecting peft==0.20.0
[2026-10-04T22:30:18.552707+00:00]   Downloading peft-0.20.0-py3-none-any.whl.metadata (14 kB)
[2026-10-04T22:30:18.566519+00:00] Requirement already satisfied: pytest in /usr/local/lib/python3.12/dist-packages (8.4.2)
[2026-10-04T22:30:18.5681

In [ ]:
import zipfile, os, pathlib, subprocess, sys
project = pathlib.Path("/content/soup-assignment")
project.mkdir(exist_ok=True)
with zipfile.ZipFile("/content/submission-bundle.zip") as archive:
    archive.extractall(project)
os.chdir(project)
print("Project:", project)
print(subprocess.check_output([sys.executable, "-m", "pytest", "tests", "-q"], text=True))
result = subprocess.run([sys.executable, "scripts/run_experiment.py", "--run-dir", "artifacts/attempt-01"])
print("Experiment process exit:", result.returncode)


Project: /content/soup-assignment
...................                                                      [100%]
19 passed in 0.34s

Experiment process exit: 0


In [ ]:
import json, pathlib
run = pathlib.Path("artifacts/attempt-01")
print([(x["name"], x["returncode"]) for x in json.loads((run / "commands.json").read_text())])
print((run / "streaming-parity.raw.log").read_text()[-12000:])
print((run / "run-summary.json").read_text())


[('nvidia-smi-before', 0), ('environment', 0), ('soup-version', 2), ('doctor', 0), ('dpo-doctor-refusal', 2), ('data-lint', 0), ('data-doctor-chat', 0), ('preflight-dry-run', 0), ('streaming-parity', 1), ('nvidia-smi-after', 0)]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 397.90it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Traceback (most recent call last):
  File "/content/soup-assignment/scripts/check_streaming.py", line 146, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/soup-assignment/scripts/check_streaming.py", line 139, in main
    result = run_check(args.device, args.dtype)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/soup-assignment/scripts/check_streaming.py", line 55, in run_check
    streamed, runtime = build_streamed_model(
                        ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/soup_cli/utils/layer_stream_runtime.py", line 2481, in build

In [ ]:
import sys, os, pathlib, zipfile, subprocess
bundle = pathlib.Path('/content/submission-bundle-v2.zip')
print('bundle bytes:', bundle.stat().st_size, 'valid zip:', zipfile.is_zipfile(bundle), flush=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall('/content/soup-assignment')
os.chdir('/content/soup-assignment')
from scripts.run_experiment import run_command
run_command([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], '/content/bootstrap-evidence', 'remove-unused-torchao')
p = pathlib.Path('scripts/check_streaming.py')
s = p.read_text().replace('attn_implementation="eager").to(device), lora)', 'attn_implementation=streamed.config._attn_implementation).to(device), lora)')
s = s.replace('dtype=dtype, device=device, runtime=runtime.stats()', 'attention_implementation=streamed.config._attn_implementation, dtype=dtype, device=device, runtime=runtime.stats()')
p.write_text(s)
run_command([sys.executable, '-m', 'pytest', '-q'], '/content/bootstrap-evidence', 'tests-attempt-02')
proc = subprocess.Popen([sys.executable, 'scripts/run_experiment.py', '--run-dir', 'artifacts/attempt-02'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
print('Evidence collector exit:', proc.wait(), flush=True)


bundle bytes: 301630 valid zip: True
[2026-10-04T22:55:57.303048Z] Found existing installation: torchao 0.10.0
[2026-10-04T22:55:57.421896Z] Uninstalling torchao-0.10.0:
[2026-10-04T22:55:59.329268Z]   Successfully uninstalled torchao-0.10.0
[2026-10-04T22:56:01.345988Z] ........................                                                 [100%]
[2026-10-04T22:56:01.346918Z] 24 passed in 0.36s
[2026-10-04T22:56:01.668401Z] Sun Oct  4 22:56:01 2026       
[2026-10-04T22:56:01.668457Z] +-----------------------------------------------------------------------------------------+
[2026-10-04T22:56:01.668477Z] | NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
[2026-10-04T22:56:01.668490Z] +-----------------------------------------+------------------------+----------------------+
[2026-10-04T22:56:01.668502Z] | GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
[2026-10-04T22:56:01.668514Z] | Fan  Temp   Perf 

In [ ]:
import pathlib, subprocess, sys
p = pathlib.Path('scripts/train_dpo.py')
s = p.read_text().replace('from pathlib import Path
', 'from pathlib import Path
import shutil
')
s = s.replace("    local_model = snapshot_download(MODEL, revision=REVISION,
                                   allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])
    write(run_dir/'model.json', {'model': MODEL, 'revision': REVISION, 'local_path': local_model})", """    snapshot = Path(snapshot_download(MODEL, revision=REVISION,
                                   allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])
    )
    # Soup's sharder rejects symlinked weights. HF snapshots use links to blobs.
    # Keep the pinned snapshot intact and materialize a regular-file checkpoint.
    materialized = Path('model-cache') / REVISION
    materialized.mkdir(parents=True, exist_ok=True)
    for source in snapshot.iterdir():
        if source.is_file():
            target = materialized / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size:
                shutil.copy2(source, target)
            assert not target.is_symlink()
    local_model = str(materialized.resolve())
    source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in materialized.iterdir() if p.is_file()}
    write(run_dir/'model.json', {'model': MODEL, 'revision': REVISION, 'local_path': local_model,
                               'materialized_source_sha256': source_hashes})""")
p.write_text(s)
compile(s, str(p), 'exec')
proc = subprocess.Popen([sys.executable, 'scripts/run_experiment.py', '--run-dir', 'artifacts/attempt-03'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
print('Evidence collector exit:', proc.wait(), flush=True)


  File "/tmp/ipykernel_279/1728153586.py", line 3
    s = p.read_text().replace('from pathlib import Path
                              ^
SyntaxError: unterminated string literal (detected at line 3)


In [ ]:
import pathlib, subprocess, sys
p = pathlib.Path('scripts/train_dpo.py')
s = p.read_text().replace('from pathlib import Path\n', 'from pathlib import Path\nimport shutil\n')
s = s.replace("    local_model = snapshot_download(MODEL, revision=REVISION,\n                                   allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])\n    write(run_dir/'model.json', {'model': MODEL, 'revision': REVISION, 'local_path': local_model})", """    snapshot = Path(snapshot_download(MODEL, revision=REVISION,
                                   allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'])
    )
    # Soup's sharder rejects symlinked weights. HF snapshots use links to blobs.
    # Keep the pinned snapshot intact and materialize a regular-file checkpoint.
    materialized = Path('model-cache') / REVISION
    materialized.mkdir(parents=True, exist_ok=True)
    for source in snapshot.iterdir():
        if source.is_file():
            target = materialized / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size:
                shutil.copy2(source, target)
            assert not target.is_symlink()
    local_model = str(materialized.resolve())
    source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in materialized.iterdir() if p.is_file()}
    write(run_dir/'model.json', {'model': MODEL, 'revision': REVISION, 'local_path': local_model,
                               'materialized_source_sha256': source_hashes})""")
p.write_text(s)
compile(s, str(p), 'exec')
print('Materialization patch saved and syntax checked.', flush=True)


Materialization patch saved and syntax checked.


In [ ]:
import pathlib, sys, subprocess, json, hashlib, shutil
p = pathlib.Path('scripts/verify_adapter.py')
s = p.read_text().replace('prefix = prefix[-max_prompt:]', 'prefix = prefix[:max_prompt]')
s = s.replace('"max_length", "max_prompt_length"}', '"max_length", "max_prompt_length", "truncation_mode"}')
s = s.replace('"max_length": args.max_length, "max_prompt_length": args.max_prompt_length}', '"max_length": args.max_length, "max_prompt_length": args.max_prompt_length, "truncation_mode": "keep_start"}')
p.write_text(s)
p = pathlib.Path('scripts/train_dpo.py')
s = p.read_text().replace('        trainer, model = wrapper.trainer, wrapper.model', '''        trainer, model = wrapper.trainer, wrapper.model
        write(run_dir/'effective-training.json', {
            'fp16': trainer.args.fp16, 'bf16': trainer.args.bf16,
            'beta': trainer.args.beta, 'loss_type': trainer.args.loss_type,
            'truncation_mode': trainer.args.truncation_mode,
            'max_length': trainer.args.max_length,
            'max_prompt_length': config.data.max_length // 2,
            'hf_gradient_checkpointing': trainer.args.gradient_checkpointing,
            'attention_implementation': model.config._attn_implementation,
            'stream_runtime': wrapper._stream_runtime.stats(),
        })''')
p.write_text(s)
for folder in ['scripts','configs','data']:
    target = pathlib.Path('artifacts/executed-source') / folder
    target.mkdir(parents=True, exist_ok=True)
    for source in pathlib.Path(folder).iterdir():
        if source.is_file():
            shutil.copy2(source, target / source.name)
print('Pinned source snapshot preserved.', flush=True)
proc = subprocess.Popen([sys.executable, 'scripts/run_experiment.py', '--run-dir', 'artifacts/attempt-03'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
print('Evidence collector exit:', proc.wait(), flush=True)


Pinned source snapshot preserved.
[2026-10-04T23:02:27.958805Z] Sun Oct  4 23:02:27 2026       
[2026-10-04T23:02:27.958875Z] +-----------------------------------------------------------------------------------------+
[2026-10-04T23:02:27.958894Z] | NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
[2026-10-04T23:02:27.958906Z] +-----------------------------------------+------------------------+----------------------+
[2026-10-04T23:02:27.958917Z] | GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
[2026-10-04T23:02:27.958927Z] | Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
[2026-10-04T23:02:27.958937Z] |                                         |                        |               MIG M. |
[2026-10-04T23:02:27.958947Z] |=========================================+========================+======================|
[2026-10-04T23:02:27.987038Z] |   0  Tesla T4     

In [ ]:
import pathlib, sys, subprocess, os, json
from scripts.run_experiment import run_command
fresh = pathlib.Path('/content/submitted-source')
run_command(['git', 'clone', 'https://github.com/Huzyefahwasim/Soup.git', str(fresh)], '/content/bootstrap-evidence', 'clone-pinned-submission')
run_command(['git', '-C', str(fresh), 'checkout', '--detach', '55bf7f4c500157a4d921cf6d9818cb3947a8f542'], '/content/bootstrap-evidence', 'checkout-pinned-submission')
os.chdir(fresh)
print('Executed repository commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(), flush=True)
run_command([sys.executable, '-m', 'pip', 'check'], '/content/bootstrap-evidence', 'pip-check')
run_command([sys.executable, '-m', 'pytest', '-q'], '/content/bootstrap-evidence', 'final-tests')
proc = subprocess.Popen([sys.executable, 'scripts/run_experiment.py', '--run-dir', 'artifacts/attempt-04'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
print('Evidence collector exit:', proc.wait(), flush=True)


[2026-10-04T23:11:28.762345Z] Cloning into '/content/submitted-source'...
[2026-10-04T23:11:29.928074Z] HEAD is now at 55bf7f4 Cover overflow logging and enforce loss and optimizer parity
Executed repository commit: 55bf7f4c500157a4d921cf6d9818cb3947a8f542
[2026-10-04T23:11:33.653699Z] ipython 7.34.0 requires jedi, which is not installed.
[2026-10-04T23:11:36.084780Z] ..........................................                               [100%]
[2026-10-04T23:11:36.086526Z] 42 passed in 0.71s
[2026-10-04T23:11:36.491758Z] Sun Oct  4 23:11:36 2026       
[2026-10-04T23:11:36.492396Z] +-----------------------------------------------------------------------------------------+
[2026-10-04T23:11:36.492430Z] | NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
[2026-10-04T23:11:36.492447Z] +-----------------------------------------+------------------------+----------------------+
[2026-10-04T23:11:36.492463Z] | GPU  Name                 Persistence-M 

In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile
from datetime import datetime, timezone

training = Path('/content/submitted-source/artifacts/attempt-04')
retry = Path('/content/verification-source/artifacts/attempt-05')
print('Original training collection (failed first verification is retained):')
print((training / 'run-summary.json').read_text())
print('Verification retry on the same saved adapter:')
print((retry / 'verification-summary.json').read_text())
print('Unchanged-adapter negative control:')
print((retry / 'negative-control.json').read_text())
print('Measured fp16 Soup shipping verdict:')
print((training / 'ship-verdict.json').read_text())

export = Path('/content/evidence-export')
export.mkdir(exist_ok=False)
for source in [Path('/content/soup-assignment/artifacts'), Path('/content/submitted-source/artifacts'), Path('/content/verification-source/artifacts')]:
    if source.exists():
        for item in source.iterdir():
            target = export / 'artifacts' / item.name
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                raise RuntimeError(f'Duplicate artifact destination: {target}')
            shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
shutil.copytree('/content/bootstrap-evidence', export / 'artifacts/bootstrap')
shutil.copy2('/content/verification-retry.raw.log', export / 'artifacts/attempt-05/outer-retry.raw.log')
for checkout, label in [('/content/submitted-source', 'training-55bf7f4'), ('/content/verification-source', 'verification-b21a349')]:
    root = Path(checkout)
    dest = export / 'executed-source' / label
    for name in ['scripts', 'configs', 'data', 'tests', 'requirements-colab.txt', 'docs']:
        source = root / name
        target = dest / name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, target, ignore=shutil.ignore_patterns('__pycache__')) if source.is_dir() else shutil.copy2(source, target)
checksums = {}
for file in sorted(export.rglob('*')):
    if file.is_file():
        if file.stat().st_size > 30 * 1024 * 1024:
            raise RuntimeError(f'Unexpected large artifact, inspect before exporting: {file}')
        checksums[str(file.relative_to(export))] = hashlib.file_digest(file.open('rb'), 'sha256').hexdigest()
manifest = dict(exported_utc=datetime.now(timezone.utc).isoformat(), files=checksums)
(export / 'export-checksums.json').write_text(json.dumps(manifest, indent=2) + '\n')
archive = Path('/content/soup-t4-evidence.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for file in sorted(export.rglob('*')):
        if file.is_file():
            zipped.write(file, file.relative_to(export))
print('Exported files:', len(checksums), 'ZIP bytes:', archive.stat().st_size)
print('Evidence archive:', archive)


Original training collection (failed first verification is retained):
{
  "finished_utc": "2026-10-04T23:41:45.866575Z",
  "domain_verdict": "DON'T SHIP",
  "parity_exit": 0,
  "training_exit": 0,
  "input_sha256": {
    "/content/submitted-source/configs/soup.yaml": "bc024a6922980cf4a78439e497db7cdde0179cb5ac650821678e78783d2e2304",
    "/content/submitted-source/data/doctor_chat.jsonl": "c108d2fa11e742173fc3ec3047547ff16fa72afb1c45f170ed2afe0a0cda31e8",
    "/content/submitted-source/data/task_eval.jsonl": "cc44e895e89b615bf03403fc21cb2edbe43f80979d1b7e450084e48e594ddde0",
    "/content/submitted-source/data/test.jsonl": "62bc5cd59dd86e3d426926a6a010667c089bfdffab37f30ac1f589cabc402f7e",
    "/content/submitted-source/data/train.jsonl": "15453a88c05eac4761cce673d979af4363392e9e7978240612985013b84c0b79",
    "/content/submitted-source/data/validation.jsonl": "2b704f98217a743aed594679ae8488b5adb83f3e823a1e627db846983c6db090"
  },
  "script_sha256": {
    "scripts/check_streaming.py": "

In [ ]:
from google.colab import files
files.download('/content/soup-t4-evidence.zip')

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display
p = Path('/content/verification-source/artifacts/attempt-05/verification.json')
encoded = base64.b64encode(p.read_bytes()).decode('ascii')
display(HTML(f'<a download="verification.json" href="data:application/json;base64,{encoded}">Download verification evidence</a>'))

Download verification evidence

In [ ]:
from IPython.display import HTML, display
import base64, hashlib
from pathlib import Path
archive = Path('/content/soup-t4-evidence.zip')
raw_archive = archive.read_bytes()
print('Archive bytes:', len(raw_archive))
print('Archive SHA-256:', hashlib.sha256(raw_archive).hexdigest())
payload = base64.b64encode(raw_archive).decode('ascii')
display(HTML(f'<a download="soup-t4-evidence.zip" href="data:application/zip;base64,{payload}">Download complete T4 evidence backup</a>'))
# The link carries the ZIP bytes so the saved output survives runtime expiry.

Archive bytes: 29878169
Archive SHA-256: c159e4504998e8f622c27c001d6dedd6785014a307b297133e2659cc04974805

Download complete T4 evidence backup